# RG3 F1/정상오탐 균형 및 선택적 앙상블
바깥평가정답 미사용, 학습 내부검증에서만 선택.

In [1]:
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
src=''.join(json.loads((root/'notebooks/29_rg3_relation_dynamics_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(src.split('records=[]')[0])
# No new sensor inputs: fixed RF std4 + RF residual4 + new relation4 CatBoost.
records=[];selection_logs=[];candidate_logs=[]
def counts(yy,pred):
    tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    return int(tp),int(fp),int(fn),int(tn),float(f1)
def cut_normal(score,yy,rate):
    normal=score[(yy==0)&np.isfinite(score)]
    return float(np.quantile(normal,1-rate,method='higher')) if len(normal) else 1.
def choose_threshold(score,yy,cap):
    best=None
    for cut in np.r_[np.unique(score),np.inf]:
        p=score>cut;tp,fp,fn,tn,f1=counts(yy,p)
        if cap is not None and fp/max((yy==0).sum(),1)>cap:continue
        key=(f1,-fp,float(cut))
        if best is None or key>best[0]:best=(key,float(cut),(tp,fp,fn,tn))
    return best
def gated(rf,cat,cut,moderate,catcut):
    return (rf>cut)|((rf>moderate)&(cat>catcut))
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        rf_outer=pair_predict(ref,q).max(axis=1)
        cat_outer=physical_predict('physical_Cat',ref,q)
        rf_inner=np.full(len(ref),np.nan);cat_inner=np.full(len(ref),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                rf_inner[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
                cat_inner[mask]=physical_predict('physical_Cat',ref[~mask],ref[mask])
        valid=np.isfinite(rf_inner)&np.isfinite(cat_inner)
        yy=y[ref][valid];ri=rf_inner[valid];ci=cat_inner[valid]
        baseline_cut=cut_normal(ri,yy,.01)
        candidates={'RF_baseline':(rf_outer>baseline_cut,dict(cut=baseline_cut))}
        for name,cap in [('RF_F1_uncapped',None),('RF_F1_FPR1',.01),('RF_F1_FPR2',.02)]:
            best=choose_threshold(ri,yy,cap);cut=best[1]
            candidates[name]=(rf_outer>cut,dict(cut=cut,inner_F1=best[0][0],inner_FP=best[2][1],cap=cap))
        choices=[]
        for high_rate in [.005,.01,.02]:
            high=cut_normal(ri,yy,high_rate)
            # RF-only route also eligible, preventing compulsory weak additions.
            tp,fp,fn,tn,f1=counts(yy,ri>high)
            choices.append(dict(high=high,moderate=np.inf,catcut=np.inf,inner_F1=f1,inner_FP=fp,high_rate=high_rate,moderate_rate=0,cat_rate=0))
            for moderate_rate in [.05,.1]:
                moderate=cut_normal(ri,yy,moderate_rate)
                for cat_rate in [.005,.01,.02]:
                    catcut=cut_normal(ci,yy,cat_rate)
                    tp,fp,fn,tn,f1=counts(yy,gated(ri,ci,high,moderate,catcut))
                    choices.append(dict(high=high,moderate=moderate,catcut=catcut,inner_F1=f1,inner_FP=fp,high_rate=high_rate,moderate_rate=moderate_rate,cat_rate=cat_rate))
        for name,cap in [('gated_F1_FPR1',.01),('gated_F1_FPR2',.02)]:
            eligible=[c for c in choices if c['inner_FP']/max((yy==0).sum(),1)<=cap]
            # Fallback to all-normal if no candidate respects cap.
            chosen=max(eligible,key=lambda c:(c['inner_F1'],-c['inner_FP'],c['moderate_rate']==0)) if eligible else dict(high=np.inf,moderate=np.inf,catcut=np.inf,inner_F1=0.,inner_FP=0,high_rate=0,moderate_rate=0,cat_rate=0)
            candidates[name]=(gated(rf_outer,cat_outer,chosen['high'],chosen['moderate'],chosen['catcut']),dict(**chosen,cap=cap))
        for candidate in choices:candidate_logs.append(dict(scheme=scheme,block=block,inner_shots=len(yy),inner_normal=int((yy==0).sum()),**candidate))
        for name,(pred,params) in candidates.items():
            selection_logs.append(dict(scheme=scheme,block=block,model=name,train_shots=len(ref),train_gas=int(y[ref].sum()),test_shots=len(q),test_gas=int(y[q].sum()),**params))
            for i,z in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),prediction=int(pred[i]),RF_score=float(rf_outer[i]),Cat_score=float(cat_outer[i])))
        print('Completed balanced',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tp,fp,fn,tn,f1=counts(g.actual,g.prediction)
    rows.append(dict(scheme=scheme,model=name,shots=len(g),gas=int(g.actual.sum()),TP=tp,FP=fp,FN=fn,TN=tn,F1=f1,precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn),FPR=fp/(fp+tn)))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_balanced_gating_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_balanced_gating_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(selection_logs).to_csv(tables/'rg3_balanced_gating_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(candidate_logs).to_csv(tables/'rg3_balanced_gating_inner_candidates.csv',index=False,encoding='utf-8-sig')
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
chk=pp.loc[pp.model.eq('RF_baseline')].merge(old.loc[old.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']],on=['scheme','TimeStamp'],validate='one_to_one')
assert len(chk)==1023 and (chk.prediction==chk.predicted_train_fpr1).all()
lines=['# RG3 가스: F1와 오탐 제한형 선택적 앙상블','',
'기존2신호RF(RF흔들림4/RF관계잔차4) max점수 및 새관계4피처CatBoost를사용한다. 정상/가스라벨을학습하는지도방식이다. 모델설정은기존유지하고판정및결합만바꾼다.',
'판정규칙은RF가강한신호이면탐지하거나,RF가중간신호일때CatBoost도강한신호일경우탐지한다. 새CatBoost단독으로정상오탐을늘리는것을줄이려는가설이다. 3개RF단독+18개선택적규칙중안쪽검증F1최대,정상오탐1/2%제한으로선택한다.',
'RF판정기준도안쪽검증F1최대(오탐제한없음/1%/2%)와기존안쪽정상99%경계로비교한다. 오탐제한은학습내부검증에만적용하며새구간오탐률을보장하지않는다. 외부평가라벨로기준/규칙을선택하지않는다.',
'바깥관측구간은학습에서제외한다. 안쪽점수도해당안쪽구간을빼고모델을학습해서생성한다. 잔차정상관계도각모델학습집합에서만학습한다. 과거평가는이후바깥구간을제외하지만안쪽검증은과거학습집합내양방향구간분리다.',
'원본이력사용조건,가스17/정상566전체유지;과거평가는가스13/정상427. 실제Run ID가아닌관측단절구간이다. 동일자료반복탐색의후보며독립최종평가아님. F1이대회공식지표인지는확인필요.',
'','| 평가 | 판정/결합 | TP | FP | FN | TN | F1 |','|---|---|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} |')
(root/'reports/rg3_balanced_gating_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(metrics.to_string(index=False))


             scheme          model  shots  gas  TP  FP  FN  TN       F1  precision   recall      FPR
leave_one_block_out    RF_baseline    583   17   4   6  13 560 0.296296   0.400000 0.235294 0.010601
leave_one_block_out RF_F1_uncapped    583   17   4  84  13 482 0.076190   0.045455 0.235294 0.148410
leave_one_block_out     RF_F1_FPR1    583   17   0   0  17 566 0.000000   0.000000 0.000000 0.000000
leave_one_block_out     RF_F1_FPR2    583   17   0   0  17 566 0.000000   0.000000 0.000000 0.000000
leave_one_block_out  gated_F1_FPR1    583   17   4   6  13 560 0.296296   0.400000 0.235294 0.010601
leave_one_block_out  gated_F1_FPR2    583   17   4   6  13 560 0.296296   0.400000 0.235294 0.010601
   past_blocks_only    RF_baseline    440   13   2   4  11 423 0.210526   0.333333 0.153846 0.009368
   past_blocks_only RF_F1_uncapped    440   13   2  32  11 395 0.085106   0.058824 0.153846 0.074941
   past_blocks_only     RF_F1_FPR1    440   13   0   0  13 427 0.000000   0.000000 0.000000